In [10]:
import os, yaml, sys
import numpy as np
import h5py
import matplotlib.pyplot as plt

ENV = os.getenv("MY_ENV", "dev")
with open("../../config.yaml", "r") as f:
    config = yaml.safe_load(f)
paths = config[ENV]["paths"]
sys.path.append(paths["src_path"])
from general_utils.utils import decode_matlab_strings

In [26]:
from dataclasses import dataclass, field

@dataclass
class Cfg:
    monkey_name: str =  'paul' # 'baby1' #
    date: str = '230204' # '220226to527' 
    img_size = 384
    folder_name = 'manyOO'  # 'fewer_occlusion' 
    batch_size = 10
cfg = Cfg()

# TODO:
- try on different datasets
- substitute it on the previous scripts (loops in image_processing.computational_models)
- scale it on the cluster and extract feats

In [6]:
a = np.load("/Users/tizianocausin/livingstone_lab_local/tiziano/models/fewer_occlusion_vit_l_16_384_blocks.0.mlp.fc2_features_meanpool.npz")["arr_0"]

In [7]:
print(a.shape)

(1024, 4379)


In [15]:

allimgs_path = f"{paths['livingstone_lab']}/tiziano/data/{cfg.monkey_name}_allimages{cfg.date}.mat"
with h5py.File(allimgs_path, "r") as f:
    try:
        refs = f["allimages"][:]      # shape (N, 1) of object refs
    except KeyError:
        refs = f["uniqueImage"][:]
    # end try:
    monkey_presentation_order = decode_matlab_strings(f, refs) # not sure if this stays the same in datasets with uniqueImage ... # ADD check
    monkey_presentation_order = sorted(set(monkey_presentation_order))


In [16]:
len(monkey_presentation_order)

4377

In [27]:
from general_utils.utils import get_relevant_output_layers
from image_processing.utils import get_usual_transform
from torch.utils.data import DataLoader
import argparse
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

transform = get_usual_transform(resize_size=cfg.img_size, normalize=False)
# task_list = get_relevant_output_layers(args.model_name, pkg=args.pkg)

dataset = ImageFolder(
    root=f"{paths['livingstone_lab']}/Stimuli/{cfg.folder_name}/",
    transform=transform,
    is_valid_file=lambda x: not x.endswith("Thumbs.db"), 
    allow_empty=True, 
)
dataloader = DataLoader(dataset, batch_size=cfg.batch_size, shuffle=False)


In [33]:
ann_presentation_order = [os.path.basename(path) for path, _ in dataset.samples] # creates the order with which images are presented to the ANN


In [36]:
ann_presentation_order

['BigAnimate_01.jpg',
 'BigAnimate_02.jpg',
 'BigAnimate_03.jpg',
 'BigAnimate_04.jpg',
 'BigAnimate_05.jpg',
 'BigAnimate_06.jpg',
 'BigAnimate_07.jpg',
 'BigAnimate_08.jpg',
 'BigAnimate_09.jpg',
 'BigAnimate_10.jpg',
 'BigAnimate_11.jpg',
 'BigAnimate_12.jpg',
 'BigAnimate_13.jpg',
 'BigAnimate_14.jpg',
 'BigAnimate_15.jpg',
 'BigAnimate_16.jpg',
 'BigAnimate_17.jpg',
 'BigAnimate_18.jpg',
 'BigAnimate_19.jpg',
 'BigAnimate_20.jpg',
 'BigAnimate_21.jpg',
 'BigAnimate_22.jpg',
 'BigAnimate_23.jpg',
 'BigAnimate_24.jpg',
 'BigAnimate_25.jpg',
 'BigAnimate_26.jpg',
 'BigAnimate_27.jpg',
 'BigAnimate_28.jpg',
 'BigAnimate_29.jpg',
 'BigAnimate_30.jpg',
 'BigAnimate_31.jpg',
 'BigAnimate_32.jpg',
 'BigAnimate_33.jpg',
 'BigAnimate_34.jpg',
 'BigAnimate_35.jpg',
 'BigAnimate_36.jpg',
 'BigAnimate_37.jpg',
 'BigAnimate_38.jpg',
 'BigAnimate_39.jpg',
 'BigAnimate_40.jpg',
 'BigAnimate_41.jpg',
 'BigAnimate_42.jpg',
 'BigAnimate_43.jpg',
 'BigAnimate_44.jpg',
 'BigAnimate_45.jpg',
 'BigAnima

In [42]:
np.savetxt("/Users/tizianocausin/livingstone_lab_local/tiziano/for_marge/manyOO_image_order.csv", ann_presentation_order, fmt="%s", delimiter=",")


In [19]:
from image_processing.computational_models import map_image_order_from_ann_to_monkey
idx = map_image_order_from_ann_to_monkey(paths, cfg.monkey_name, cfg.date, dataset)
# then use this to load and index the features

In [21]:
feats_new = a[:, idx]

In [ ]:
cfg.model_name = 'vit_l_16'; cfg.img_size = 384; cfg.pooling = 'mean'; cfg.pkg = 'timm';
feats_filename = f"{paths['livingstone_lab']}/tiziano/models/{cfg.monkey_name}_{cfg.date}_{cfg.model_name}_{cfg.img_size}_blocks.0.mlp.fc2_features_{cfg.pooling}pool.npz"
feats_old = np.load(feats_filename)["arr_0"]


In [25]:
np.allclose(feats_old, feats_new)

True

In [ ]:
act = []
for i in idx:
    act.append(os.path.basename(dataset.samples[i][0]))
    # print((base_dataset.samples[i]))

In [66]:
act == monkey_presentation_order

True